# 1. Configuração do Ambiente e Ingestão de Dados

Nesta primeira etapa, instalamos a biblioteca `lasio` e preparamos o ecossistema do Google Colab para renderizar os widgets interativos do Plotly em tempo real (através da função `enable_custom_widget_manager`). A rotina `carregamento_pocos` varre o diretório especificado, lê todos os arquivos `.las` e os armazena na memória em um dicionário estruturado de DataFrames, pronto para o cruzamento de dados petrofísicos.

In [1]:
# Instalação da dependência
!pip install lasio -q

import pandas as pd
import numpy as np
import os
import lasio
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from google.colab import output

# Habilita a renderização de widgets interativos (essencial para o Lasso funcionar no Colab)
output.enable_custom_widget_manager()

def carregamento_pocos():
    """Carrega os arquivos LAS do diretório para um dicionário de DataFrames."""
    pasta = "/content/drive/MyDrive/poços_sapinhoa"
    dicionario_pocos = {}

    if not os.path.exists(pasta):
        print(f"Diretório não encontrado: {pasta}")
        return dicionario_pocos

    arquivos = [f for f in os.listdir(pasta) if f.lower().endswith(".las")]

    for arquivo in arquivos:
        caminho_arquivo = os.path.join(pasta, arquivo)
        try:
            las = lasio.read(caminho_arquivo)
            dicionario_pocos[arquivo] = las.df().reset_index()
        except Exception as e:
            print(f"Erro ao ler {arquivo}: {e}")

    return dicionario_pocos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 1.4 MB/s eta 0:00:00


# 2. Motor do Dashboard Interativo (Plotly + Ipywidgets)

Este bloco contém a lógica da ferramenta analítica. A função `dashboard_interativo` foi arquitetada para garantir alta performance:
*   **Limpeza Automática:** Remove valores nulos (ex: `-9999`) antes da plotagem. Isso impede desalinhamento de índices entre o crossplot e o perfil durante a seleção.
*   **Pré-alocação (FigureWidget):** Os gráficos não são reconstruídos a cada seleção. O script cria *traces* (camadas) de marcação transparentes logo no início.
*   **Atualização Atômica (`batch_update`):** Quando a seleção (Lasso/Box) ocorre, a função extrai as coordenadas dos dados de interesse e injeta os valores nas camadas pré-alocadas de uma única vez. Isso impede o travamento do navegador e a duplicação dos gráficos.
*   **Layout Nativo:** Substituímos a injeção em HTML por `HBox` e `VBox` do pacote `ipywidgets`, garantindo um alinhamento lateral limpo entre o *scatter* e o *well log*.

In [4]:
from plotly.subplots import make_subplots

def dashboard_interativo(dicionario_pocos):
    # 1. Validações de Entrada e Seleção de Curvas
    nome_arquivo = input("Digite o nome do arquivo (ex: poço_A.las): ")
    if nome_arquivo not in dicionario_pocos:
        print(f"Erro: Arquivo '{nome_arquivo}' não encontrado no dicionário.")
        return

    df_bruto = dicionario_pocos[nome_arquivo]
    profundidade = 'DEPTH'

    if profundidade not in df_bruto.columns:
        print(f"Erro: Coluna '{profundidade}' não encontrada.")
        return

    valor_track1 = input('Digite a primeira curva para o eixo X (ex: RHOB): ')
    valor_track2 = input('Digite a segunda curva para o eixo Y (ex: NPHI): ')

    if valor_track1 not in df_bruto.columns or valor_track2 not in df_bruto.columns:
        print("Erro: Uma das curvas não foi encontrada no arquivo.")
        return

    # 2. Tratamento de Dados (Limpeza de nulos para alinhamento)
    df = df_bruto.replace(-9999, np.nan).dropna(subset=[profundidade, valor_track1, valor_track2]).reset_index(drop=True)

    # 3. Criação da Janela Unificada (Subplots)
    # Em vez de dois gráficos separados, criamos 1 painel com 2 colunas
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('Crossplot (Selecione com Lasso/Box)', 'Perfil de Poço'),
                        horizontal_spacing=0.15)

    # Camada 0: Crossplot (Scatter) -> Coluna 1
    fig.add_trace(go.Scatter(x=df[valor_track1], y=df[valor_track2], mode='markers', name='Amostras',
                             marker=dict(color='gray', opacity=0.4, size=5)), row=1, col=1)

    # Camada 1 e 2: Linhas do Perfil -> Coluna 2
    fig.add_trace(go.Scatter(x=df[valor_track1], y=df[profundidade], mode='lines', name=valor_track1,
                             line=dict(color='#1f77b4', width=1.5)), row=1, col=2)
    fig.add_trace(go.Scatter(x=df[valor_track2], y=df[profundidade], mode='lines', name=valor_track2,
                             line=dict(color='#d62728', width=1.5)), row=1, col=2)

    # Camada 3 e 4: Marcadores de Seleção (Pré-alocados vazios) -> Coluna 2
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', name=f'Sel. {valor_track1}',
                             marker=dict(color='cyan', size=8, symbol='circle-open', line=dict(width=2))), row=1, col=2)
    fig.add_trace(go.Scatter(x=[], y=[], mode='markers', name=f'Sel. {valor_track2}',
                             marker=dict(color='magenta', size=8, symbol='circle-open', line=dict(width=2))), row=1, col=2)

    # 4. Ajustes de Layout Específicos
    fig.update_layout(height=800, width=1100, dragmode='lasso', hovermode='closest')

    # Adiciona os títulos dos eixos X para ambos os gráficos
    fig.update_xaxes(title_text=valor_track1, row=1, col=1)
    fig.update_xaxes(title_text="Amplitude", row=1, col=2)

    # Inverte apenas o eixo Y do gráfico da direita (Perfil de Poço)
    fig.update_yaxes(title_text=valor_track2, row=1, col=1)
    fig.update_yaxes(title_text="Profundidade (m)", autorange='reversed', row=1, col=2)

    # 5. Converte a figura nativa para o formato interativo (FigureWidget)
    fw = go.FigureWidget(fig)

    # 6. Lógica de Atualização (Síncrona)
    def ao_selecionar_pontos(trace, points, selector):
        indices = points.point_inds
        if not indices:
            return

        # Preenche as Camadas 3 e 4 na Coluna 2 com as amostras selecionadas na Coluna 1
        with fw.batch_update():
            fw.data[3].x = df.loc[indices, valor_track1]
            fw.data[3].y = df.loc[indices, profundidade]

            fw.data[4].x = df.loc[indices, valor_track2]
            fw.data[4].y = df.loc[indices, profundidade]

    # Atrela o evento de seleção EXCLUSIVAMENTE à Camada 0 (Crossplot)
    fw.data[0].on_selection(ao_selecionar_pontos)

    # 7. Botão de Limpeza
    btn_limpar = widgets.Button(description="Limpar Seleção", button_style='warning', icon='trash')

    def limpar_graficos(b):
        with fw.batch_update():
            fw.data[3].x = []
            fw.data[3].y = []
            fw.data[4].x = []
            fw.data[4].y = []

    btn_limpar.on_click(limpar_graficos)

    # Renderiza o Botão e, logo abaixo, o FigureWidget unificado
    display(btn_limpar, fw)

# 3. Execução da Interface

Com os diretórios configurados e a lógica instanciada, este bloco invoca o carregamento da base de dados e aciona o painel. Siga as instruções que surgirão no console para definir o poço alvo e quais variáveis comporão os eixos X e Y.

In [5]:
# 1. Carrega todos os poços do diretório para a memória
print("Carregando arquivos LAS da bacia...")
dicionario_base = carregamento_pocos()

# 2. Inicia o painel
if dicionario_base:
    print("-" * 40)
    dashboard_interativo(dicionario_base)

Carregando arquivos LAS da bacia...
----------------------------------------
Digite o nome do arquivo (ex: poço_A.las): 9-SPS-95-SP_BASE.las
Digite a primeira curva para o eixo X (ex: RHOB): DWAL
Digite a segunda curva para o eixo Y (ex: NPHI): DWCA


Button(button_style='warning', description='Limpar Seleção', icon='trash', style=ButtonStyle())

FigureWidget({
    'data': [{'marker': {'color': 'gray', 'opacity': 0.4, 'size': 5},
              'mode': 'markers',
              'name': 'Amostras',
              'type': 'scatter',
              'uid': '97514556-edf1-419b-97f9-de43f23981c2',
              'x': array([0.01 , 0.008, 0.   , ..., 0.   , 0.   , 0.   ]),
              'xaxis': 'x',
              'y': array([0.296, 0.277, 0.231, ..., 0.385, 0.385, 0.385]),
              'yaxis': 'y'},
             {'line': {'color': '#1f77b4', 'width': 1.5},
              'mode': 'lines',
              'name': 'DWAL',
              'type': 'scatter',
              'uid': 'aa8b05dd-eafa-496a-9a9a-c1b17b69fa88',
              'x': array([0.01 , 0.008, 0.   , ..., 0.   , 0.   , 0.   ]),
              'xaxis': 'x2',
              'y': array([5112.046, 5112.199, 5112.351, ..., 6038.334, 6038.486, 6038.638]),
              'yaxis': 'y2'},
             {'line': {'color': '#d62728', 'width': 1.5},
              'mode': 'lines',
              'nam